# Workshop 05 — Properties, Buying, and Rent

This is where the game gets strategic! In this workshop you will add the ability to buy properties, track who owns what, and charge rent when a player lands on someone else's property. You will also handle tax spaces and the "GO TO JAIL" square.

**What you will learn:**
- The `switch` statement for handling different space types
- Searching arrays to find property owners
- Adding a "buy" phase to the turn system
- Checking for bankruptcy (game over)

**Time:** roughly 15 to 20 minutes.

## Section 1 — The handleSpace Function

When a player lands on a space, different things happen depending on the space type. A `switch` statement is the cleanest way to handle this. Think of it like a series of `if/else if` blocks, but more readable.

In [ ]:
import os

project_folder = os.path.join(os.path.expanduser("~"), "text_monopoly")
os.makedirs(project_folder, exist_ok=True)

# This cell shows the handleSpace logic in isolation.
# We will integrate it into the full game in Section 3.

print("""The handleSpace function uses a switch statement:

switch (space.type) {
    case "go":
        // Player collects £200
        break;

    case "property":
        // Check if owned, offer to buy, or charge rent
        break;

    case "tax":
        // Deduct tax amount
        break;

    case "gotojail":
        // Send player to jail
        break;

    case "chance":
        // Draw a random chance card
        break;

    case "chest":
        // Draw a random community chest card
        break;

    default:
        // Free parking, just visiting - nothing happens
        break;
}

Each 'case' handles one type of space.
The 'break' keyword stops execution falling through to the next case.""")

## Section 2 — Finding the Property Owner

When a player lands on a property, we need to check whether anyone owns it. We do this by searching through every player's `properties` array to see if the board position appears in it.

In [ ]:
print("""The getPropertyOwner function:

function getPropertyOwner(position) {
    for (let i = 0; i < state.players.length; i++) {
        if (state.players[i].properties.includes(position)) {
            return i;  // return the player index (0 or 1)
        }
    }
    return null;  // nobody owns it
}

Three possible outcomes when landing on a property:
1. owner === null       -> unowned, offer to buy
2. owner === current    -> you own it, nothing happens
3. owner === other      -> pay rent to the owner""")

## Section 3 — The Complete Game with Properties

This is a big step. The code below includes everything from Workshop 04 plus buying, rent, tax, jail, and chance/community chest cards. Take your time reading through it.

In [ ]:
html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Text Monopoly - Properties</title>
    <style>
        :root {
            --bg: #0d1117; --surface: #161b22; --border: #30363d;
            --text: #e6edf3; --dim: #8b949e; --green: #3fb950;
            --blue: #58a6ff; --red: #f85149; --yellow: #d29922; --purple: #bc8cff;
        }
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { background: var(--bg); color: var(--text); font-family: Georgia, serif; padding: 20px; }
        h1 { color: var(--green); margin-bottom: 4px; }
        .subtitle { color: var(--dim); font-size: 0.85rem; margin-bottom: 20px; }
        .game-container { display: grid; grid-template-columns: 1fr 320px; gap: 20px; max-width: 1100px; }
        .console { background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 16px; height: 520px; overflow-y: auto; font-size: 0.82rem; line-height: 1.7; }
        .sidebar { display: flex; flex-direction: column; gap: 12px; }
        .card { background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 16px; }
        .card h3 { margin-bottom: 8px; }
        .stat { display: flex; justify-content: space-between; padding: 3px 0; font-size: 0.85rem; }
        .label { color: var(--dim); }
        .props { margin-top: 8px; font-size: 0.75rem; color: var(--dim); }
        .props span { display: inline-block; background: var(--bg); border: 1px solid var(--border); border-radius: 4px; padding: 2px 8px; margin: 2px; color: var(--text); }
        .dice-display { text-align: center; font-size: 2.5rem; padding: 8px; letter-spacing: 12px; }
        .turn-indicator { text-align: center; font-size: 0.85rem; padding: 8px; border-radius: 8px; font-weight: bold; }
        .turn-p1 { background: rgba(63, 185, 80, 0.15); color: var(--green); }
        .turn-p2 { background: rgba(88, 166, 255, 0.15); color: var(--blue); }
        button { width: 100%; padding: 12px; border: 1px solid var(--border); border-radius: 8px; background: var(--bg); color: var(--text); font-size: 0.9rem; cursor: pointer; margin-bottom: 8px; font-weight: bold; }
        button:hover:not(:disabled) { border-color: var(--green); }
        button:disabled { opacity: 0.3; cursor: not-allowed; }
        .roll-btn { background: var(--green); border: none; color: #fff; }
        .log-system { color: var(--dim); } .log-player1 { color: var(--green); }
        .log-player2 { color: var(--blue); } .log-event { color: var(--yellow); }
        .log-danger { color: var(--red); } .log-reward { color: var(--purple); }
    </style>
</head>
<body>
    <h1>Text Monopoly</h1>
    <p class="subtitle">Workshop 05 - properties, buying, and rent</p>
    <div class="game-container">
        <div class="console" id="console"></div>
        <div class="sidebar">
            <div class="turn-indicator" id="turnIndicator">Player 1's Turn</div>
            <div class="dice-display" id="diceDisplay">🎲 🎲</div>
            <div class="card" id="p1card"></div>
            <div class="card" id="p2card"></div>
            <div class="card">
                <button class="roll-btn" id="rollBtn" onclick="rollDice()">🎲 Roll Dice</button>
                <button id="buyBtn" onclick="buyProperty()" disabled>Buy Property</button>
                <button id="endTurnBtn" onclick="endTurn()" disabled>End Turn</button>
            </div>
        </div>
    </div>

    <script>
        const BOARD = [
            { name: "GO", type: "go" },
            { name: "Old Kent Road", type: "property", cost: 60, rent: 10, color: "#8B4513" },
            { name: "Community Chest", type: "chest" },
            { name: "Whitechapel Rd", type: "property", cost: 60, rent: 10, color: "#8B4513" },
            { name: "Income Tax", type: "tax", amount: 200 },
            { name: "Kings Cross", type: "property", cost: 200, rent: 50, color: "#333" },
            { name: "Angel Islington", type: "property", cost: 100, rent: 20, color: "#87CEEB" },
            { name: "Chance", type: "chance" },
            { name: "Euston Road", type: "property", cost: 100, rent: 20, color: "#87CEEB" },
            { name: "Pentonville Rd", type: "property", cost: 120, rent: 25, color: "#87CEEB" },
            { name: "Just Visiting", type: "visiting" },
            { name: "Pall Mall", type: "property", cost: 140, rent: 30, color: "#FF00FF" },
            { name: "Electric Co.", type: "property", cost: 150, rent: 35, color: "#FFD700" },
            { name: "Whitehall", type: "property", cost: 140, rent: 30, color: "#FF00FF" },
            { name: "Northumberland", type: "property", cost: 160, rent: 35, color: "#FF00FF" },
            { name: "Marylebone Stn", type: "property", cost: 200, rent: 50, color: "#333" },
            { name: "Bow Street", type: "property", cost: 180, rent: 40, color: "#FF8C00" },
            { name: "Community Chest", type: "chest" },
            { name: "Marlborough St", type: "property", cost: 180, rent: 40, color: "#FF8C00" },
            { name: "Vine Street", type: "property", cost: 200, rent: 45, color: "#FF8C00" },
            { name: "Free Parking", type: "free" },
            { name: "Strand", type: "property", cost: 220, rent: 50, color: "#FF0000" },
            { name: "Chance", type: "chance" },
            { name: "Fleet Street", type: "property", cost: 220, rent: 50, color: "#FF0000" },
            { name: "Trafalgar Sq", type: "property", cost: 240, rent: 55, color: "#FF0000" },
            { name: "Fenchurch Stn", type: "property", cost: 200, rent: 50, color: "#333" },
            { name: "Leicester Sq", type: "property", cost: 260, rent: 60, color: "#FFFF00" },
            { name: "Coventry Street", type: "property", cost: 260, rent: 60, color: "#FFFF00" },
            { name: "Water Works", type: "property", cost: 150, rent: 35, color: "#FFD700" },
            { name: "Piccadilly", type: "property", cost: 280, rent: 65, color: "#FFFF00" },
            { name: "GO TO JAIL", type: "gotojail" },
            { name: "Regent Street", type: "property", cost: 300, rent: 70, color: "#006400" },
            { name: "Oxford Street", type: "property", cost: 300, rent: 70, color: "#006400" },
            { name: "Community Chest", type: "chest" },
            { name: "Bond Street", type: "property", cost: 320, rent: 75, color: "#006400" },
            { name: "Liverpool Stn", type: "property", cost: 200, rent: 50, color: "#333" },
            { name: "Chance", type: "chance" },
            { name: "Park Lane", type: "property", cost: 350, rent: 85, color: "#00008B" },
            { name: "Super Tax", type: "tax", amount: 100 },
            { name: "Mayfair", type: "property", cost: 400, rent: 100, color: "#00008B" }
        ];

        const CHANCE_CARDS = [
            { text: "Advance to GO! Collect £200", action: function(p) { p.position = 0; p.cash += 200; } },
            { text: "Bank pays you dividend of £50", action: function(p) { p.cash += 50; } },
            { text: "Go back 3 spaces", action: function(p) { p.position = (p.position - 3 + 40) % 40; } },
            { text: "Go directly to Jail!", action: function(p) { p.position = 10; p.inJail = true; p.jailTurns = 0; } },
            { text: "You won a crossword! Collect £100", action: function(p) { p.cash += 100; } },
            { text: "Speeding fine: pay £50", action: function(p) { p.cash -= 50; } }
        ];

        const CHEST_CARDS = [
            { text: "Bank error in your favour! Collect £200", action: function(p) { p.cash += 200; } },
            { text: "Doctor's fees: pay £50", action: function(p) { p.cash -= 50; } },
            { text: "Holiday fund matures: collect £100", action: function(p) { p.cash += 100; } },
            { text: "Pay school fees of £150", action: function(p) { p.cash -= 150; } },
            { text: "Income tax refund: collect £20", action: function(p) { p.cash += 20; } },
            { text: "Go to Jail!", action: function(p) { p.position = 10; p.inJail = true; p.jailTurns = 0; } }
        ];

        let state = {
            players: [
                { name: "Player 1", cash: 1500, position: 0, properties: [], inJail: false, jailTurns: 0 },
                { name: "Player 2", cash: 1500, position: 0, properties: [], inJail: false, jailTurns: 0 }
            ],
            currentPlayer: 0, dice: [0, 0], phase: "roll", gameOver: false, turnCount: 0
        };

        function log(text, cls) {
            const el = document.getElementById("console");
            const line = document.createElement("div");
            line.className = "log-" + (cls || "system");
            line.textContent = "> " + text;
            el.appendChild(line);
            el.scrollTop = el.scrollHeight;
        }

        function getPropertyOwner(pos) {
            for (let i = 0; i < state.players.length; i++) {
                if (state.players[i].properties.includes(pos)) return i;
            }
            return null;
        }

        function checkBankrupt(p) {
            if (p.cash <= 0) {
                state.gameOver = true;
                const winner = state.players.find(function(pl) { return pl !== p; });
                log(p.name + " is BANKRUPT!", "danger");
                log(winner.name + " WINS with £" + winner.cash + "!", "reward");
            }
        }

        function handleSpace(p, space, pClass) {
            switch (space.type) {
                case "go":
                    p.cash += 200;
                    log("Landed on GO! Collect £200", "reward");
                    state.phase = "endturn";
                    break;

                case "property":
                    var owner = getPropertyOwner(p.position);
                    if (owner === null) {
                        if (p.cash >= space.cost) {
                            state.phase = "buy";
                            log(space.name + " is available for £" + space.cost + ". Buy it?", "event");
                        } else {
                            log("Cannot afford " + space.name + " (£" + space.cost + ")", "system");
                            state.phase = "endturn";
                        }
                    } else if (owner !== state.currentPlayer) {
                        p.cash -= space.rent;
                        state.players[owner].cash += space.rent;
                        log("Owned by " + state.players[owner].name + "! Pay £" + space.rent + " rent", "danger");
                        state.phase = "endturn";
                        checkBankrupt(p);
                    } else {
                        log("You own this property", "system");
                        state.phase = "endturn";
                    }
                    break;

                case "tax":
                    p.cash -= space.amount;
                    log("Tax! Pay £" + space.amount, "danger");
                    state.phase = "endturn";
                    checkBankrupt(p);
                    break;

                case "gotojail":
                    p.position = 10;
                    p.inJail = true;
                    p.jailTurns = 0;
                    log("GO TO JAIL!", "danger");
                    state.phase = "endturn";
                    break;

                case "chance":
                    var card = CHANCE_CARDS[Math.floor(Math.random() * CHANCE_CARDS.length)];
                    log("Chance: " + card.text, "event");
                    card.action(p);
                    state.phase = "endturn";
                    checkBankrupt(p);
                    break;

                case "chest":
                    var card2 = CHEST_CARDS[Math.floor(Math.random() * CHEST_CARDS.length)];
                    log("Community Chest: " + card2.text, "event");
                    card2.action(p);
                    state.phase = "endturn";
                    checkBankrupt(p);
                    break;

                default:
                    state.phase = "endturn";
                    break;
            }
        }

        function rollDice() {
            if (state.phase !== "roll" || state.gameOver) return;
            var d1 = Math.floor(Math.random() * 6) + 1;
            var d2 = Math.floor(Math.random() * 6) + 1;
            state.dice = [d1, d2];
            var total = d1 + d2;
            var diceEmoji = ['', '⚀', '⚁', '⚂', '⚃', '⚄', '⚅'];
            document.getElementById("diceDisplay").textContent = diceEmoji[d1] + " " + diceEmoji[d2];

            var p = state.players[state.currentPlayer];
            var pClass = state.currentPlayer === 0 ? "player1" : "player2";
            log(p.name + " rolled " + d1 + " + " + d2 + " = " + total, pClass);

            // Jail logic
            if (p.inJail) {
                p.jailTurns++;
                if (d1 === d2) { p.inJail = false; log("Doubles! Escaped jail!", "event"); }
                else if (p.jailTurns >= 3) { p.inJail = false; p.cash -= 50; log("Paid £50 fine to leave jail", "danger"); }
                else { log("Still in jail (" + p.jailTurns + "/3)", "danger"); state.phase = "endturn"; updateUI(); return; }
            }

            var oldPos = p.position;
            p.position = (p.position + total) % 40;
            if (p.position < oldPos && p.position !== 0) { p.cash += 200; log("Passed GO! Collect £200", "reward"); }

            var space = BOARD[p.position];
            log("Landed on: " + space.name, pClass);
            handleSpace(p, space, pClass);
            updateUI();
        }

        function buyProperty() {
            if (state.phase !== "buy") return;
            var p = state.players[state.currentPlayer];
            var space = BOARD[p.position];
            p.cash -= space.cost;
            p.properties.push(p.position);
            log(p.name + " bought " + space.name + " for £" + space.cost + "!", "reward");
            state.phase = "endturn";
            updateUI();
        }

        function endTurn() {
            if (state.phase !== "buy" && state.phase !== "endturn") return;
            state.currentPlayer = (state.currentPlayer + 1) % 2;
            state.phase = "roll";
            state.turnCount++;
            updateUI();
        }

        function updateUI() {
            var cp = state.currentPlayer;
            var ti = document.getElementById("turnIndicator");
            ti.textContent = state.players[cp].name + "'s Turn";
            ti.className = "turn-indicator turn-p" + (cp + 1);

            var colours = ["var(--green)", "var(--blue)"];
            for (var i = 0; i < 2; i++) {
                var pl = state.players[i];
                var propsHtml = "No properties yet";
                if (pl.properties.length > 0) {
                    propsHtml = pl.properties.map(function(pos) {
                        return "<span>" + BOARD[pos].name + "</span>";
                    }).join("");
                }
                document.getElementById("p" + (i+1) + "card").innerHTML =
                    "<h3 style='color:" + colours[i] + "'>" + pl.name + "</h3>" +
                    "<div class='stat'><span class='label'>Cash</span><span>£" + pl.cash + "</span></div>" +
                    "<div class='stat'><span class='label'>Position</span><span>" + BOARD[pl.position].name + "</span></div>" +
                    "<div class='stat'><span class='label'>In Jail</span><span>" + (pl.inJail ? "Yes (" + pl.jailTurns + "/3)" : "No") + "</span></div>" +
                    "<div class='props'>" + propsHtml + "</div>";
            }

            document.getElementById("rollBtn").disabled = state.phase !== "roll" || state.gameOver;
            document.getElementById("buyBtn").disabled = state.phase !== "buy" || state.gameOver;
            document.getElementById("endTurnBtn").disabled = (state.phase !== "endturn" && state.phase !== "buy") || state.gameOver;
        }

        log("Welcome to Text Monopoly! 🎩");
        log("Each player starts with £1500.");
        log("Roll the dice to begin!");
        updateUI();
    </script>
</body>
</html>
"""

filepath = os.path.join(project_folder, "05_properties_and_rent.html")
with open(filepath, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"File saved: {filepath}")
print("This is a fully playable game! Buy properties and charge rent.")

## Section 4 — The Three-Phase Turn

The turn now has three possible phases:

| Phase | Meaning | Available actions |
|---|---|---|
| `"roll"` | Waiting for dice roll | Roll Dice button |
| `"buy"` | Landed on unowned property | Buy Property or End Turn |
| `"endturn"` | Turn is finishing | End Turn button |

When a player lands on an unowned property they can afford, the phase becomes `"buy"`. They can then choose to buy it or skip by pressing End Turn.

## Section 5 — Challenge

1. The game currently has no winner banner. Add a visible message when someone goes bankrupt (hint: create a new `<div>` and set its `textContent`).
2. Add a "Building repairs: pay £150" chance card to the `CHANCE_CARDS` array.
3. Try adding a simple rule: if a player owns both properties of the same colour, rent is doubled. (This is tricky but rewarding to figure out!)

In **Workshop 06** you will add the jail system with doubles and the three-turn escape rule. 🏠